<img src="https://github.com/comet-ml/opik/blob/main/apps/opik-documentation/documentation/static/img/opik-logo.svg?raw=true" width="200" height="100" alt="Opik Logo">

# Opik Model Evaluation Workshop

## **Overview**
This session provides a brief introduction into **tracing** your model calls with Opik as well as running **experiments** on standardized **datasets** to see how different models compare. In this session you will:
- Configure Opik with your API Key and Workspace
- Build a simple Netflix Q&A application using OpenAI
- Trace all model calls in Opik
- Create a dataset of user inputs and ground truth expected answers for questions on the Netflix catalogue
- Run experiments on the dataset with 3 different models/prompts
- Compare the performance of the three different models with leaderboard and custom charts

## Setup & Configuration

In [ ]:
%pip install opik openai --q

Configure Opik with your API Key and personal workspace

In [ ]:
import opik
opik.configure(use_local=False, force=True)

In [ ]:
import os
import json
import getpass
from opik.integrations.openai import track_openai
from opik.evaluation import evaluate
from opik.evaluation.metrics import Hallucination, AnswerRelevance, Usefulness, LevenshteinRatio
import openai

PROJECT_NAME = "Netflix-Model-Evaluation"

#Optional: setup custom OppenAI compatible model provider to use as the LLM juge
# os.environ["OPENAI_BASE_URL"] = "https://api.your-provider.com/v1"

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

openai_client = track_openai(openai.Client())

## Defining the Application

### Single Model Call: Netflix Content Q&A

This demo evaluates a **single model call** that answers questions about the Netflix content catalog. There is no agent, no routing, no tool use — just a direct call to an OpenAI model with a system prompt and catalog context.

We will compare **3 different model + prompt configurations** to see how model capability and prompt engineering affect response quality:

1. **gpt-4o-mini** with a minimal prompt — cheapest model, bare-bones instructions
2. **gpt-4o** with a standard prompt — mid-tier model, structured guidelines
3. **gpt-4.1** with an expert-crafted prompt — most capable model, detailed instructions

While we use OpenAI models for demo simplicity purposes, these can be replaces with any other off the shelf or fine-tuned models of your choice. 

In [ ]:
NETFLIX_CATALOG = json.dumps([
    {
        "title": "Stranger Things",
        "type": "Series",
        "genre": ["Sci-Fi", "Horror", "Drama"],
        "seasons": 5,
        "year_released": 2016,
        "year_ended": 2025,
        "rating": "TV-14",
        "imdb_score": 8.7,
        "cast": ["Millie Bobby Brown", "Finn Wolfhard", "Winona Ryder", "David Harbour"],
        "synopsis": "When a young boy vanishes, a small town uncovers a mystery involving secret experiments, terrifying supernatural forces, and one strange little girl.",
        "total_episodes": 42,
        "avg_episode_length_min": 50
    },
    {
        "title": "Squid Game",
        "type": "Series",
        "genre": ["Thriller", "Drama", "Action"],
        "seasons": 2,
        "year_released": 2021,
        "year_ended": None,
        "rating": "TV-MA",
        "imdb_score": 8.0,
        "cast": ["Lee Jung-jae", "Park Hae-soo", "Wi Ha-jun", "HoYeon Jung"],
        "synopsis": "Hundreds of cash-strapped players accept a strange invitation to compete in children's games. Inside, a tempting prize awaits with deadly high stakes.",
        "total_episodes": 16,
        "avg_episode_length_min": 55
    },
    {
        "title": "Wednesday",
        "type": "Series",
        "genre": ["Comedy", "Horror", "Mystery"],
        "seasons": 2,
        "year_released": 2022,
        "year_ended": None,
        "rating": "TV-14",
        "imdb_score": 8.1,
        "cast": ["Jenna Ortega", "Gwendoline Christie", "Riki Lindhome", "Catherine Zeta-Jones"],
        "synopsis": "Smart, sarcastic and a little dead inside, Wednesday Addams investigates a murder spree while navigating new relationships at Nevermore Academy.",
        "total_episodes": 16,
        "avg_episode_length_min": 45
    },
    {
        "title": "The Crown",
        "type": "Series",
        "genre": ["Drama", "Historical", "Biography"],
        "seasons": 6,
        "year_released": 2016,
        "year_ended": 2023,
        "rating": "TV-MA",
        "imdb_score": 8.6,
        "cast": ["Claire Foy", "Olivia Colman", "Imelda Staunton", "Matt Smith"],
        "synopsis": "Follows the political rivalries and romance of Queen Elizabeth II's reign and the events that shaped the second half of the twentieth century.",
        "total_episodes": 60,
        "avg_episode_length_min": 58
    },
    {
        "title": "The Queen's Gambit",
        "type": "Limited Series",
        "genre": ["Drama"],
        "seasons": 1,
        "year_released": 2020,
        "year_ended": 2020,
        "rating": "TV-MA",
        "imdb_score": 8.5,
        "cast": ["Anya Taylor-Joy", "Bill Camp", "Thomas Brodie-Sangster"],
        "synopsis": "Orphaned at age nine, prodigious introvert Beth Harmon discovers and masters the game of chess in 1960s USA while struggling with addiction.",
        "total_episodes": 7,
        "avg_episode_length_min": 56
    },
    {
        "title": "Glass Onion: A Knives Out Mystery",
        "type": "Film",
        "genre": ["Mystery", "Comedy", "Crime"],
        "year_released": 2022,
        "rating": "PG-13",
        "imdb_score": 7.1,
        "cast": ["Daniel Craig", "Edward Norton", "Janelle Monáe", "Kate Hudson"],
        "synopsis": "Tech billionaire Miles Bron invites his friends for a getaway on his private Greek island, but when someone turns up dead, Detective Blanc investigates.",
        "runtime_min": 139
    },
    {
        "title": "All Quiet on the Western Front",
        "type": "Film",
        "genre": ["War", "Drama", "Action"],
        "year_released": 2022,
        "rating": "R",
        "imdb_score": 7.8,
        "cast": ["Felix Kammerer", "Albrecht Schuch", "Aaron Hilmer"],
        "synopsis": "A young German soldier on the Western Front of World War I experiences the dehumanizing horrors of warfare, shattering his initial patriotic idealism.",
        "runtime_min": 148,
        "awards": ["4 Academy Awards including Best International Feature Film"]
    },
    {
        "title": "Bridgerton",
        "type": "Series",
        "genre": ["Romance", "Drama", "Historical"],
        "seasons": 3,
        "year_released": 2020,
        "year_ended": None,
        "rating": "TV-MA",
        "imdb_score": 7.3,
        "cast": ["Adjoa Andoh", "Nicola Coughlan", "Luke Newton", "Jonathan Bailey"],
        "synopsis": "Set in the Regency era, the eight Bridgerton siblings navigate London's high society in search of love, drama, and independence.",
        "total_episodes": 24,
        "avg_episode_length_min": 60
    },
    {
        "title": "Black Mirror",
        "type": "Series",
        "genre": ["Sci-Fi", "Thriller", "Drama"],
        "seasons": 7,
        "year_released": 2011,
        "year_ended": None,
        "rating": "TV-MA",
        "imdb_score": 8.7,
        "cast": ["Various (anthology)"],
        "synopsis": "An anthology series exploring a twisted, high-tech multiverse where humanity's greatest innovations and darkest instincts collide.",
        "total_episodes": 27,
        "avg_episode_length_min": 55
    },
    {
        "title": "Dark",
        "type": "Series",
        "genre": ["Sci-Fi", "Thriller", "Mystery"],
        "seasons": 3,
        "year_released": 2017,
        "year_ended": 2020,
        "rating": "TV-MA",
        "imdb_score": 8.8,
        "cast": ["Louis Hofmann", "Oliver Masucci", "Jördis Triebel"],
        "synopsis": "A family saga with a supernatural twist, set in a German town where the disappearance of two children exposes the relationships among four families.",
        "total_episodes": 26,
        "avg_episode_length_min": 52
    }
], indent=2)

In [ ]:
from opik import opik_context

@opik.track(project_name=PROJECT_NAME)
def answer_question(question: str, model: str, system_prompt: str) -> str:
    opik_context.update_current_trace(prompts=[system_prompt])
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": f"{system_prompt.prompt}\n\n## Netflix Catalog\n{NETFLIX_CATALOG}"
            },
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content.strip()

## Test the Application

Run a single query to verify the application works and see the trace in Opik.

In [ ]:
test_prompt = opik.Prompt(
    name="Netflix Q&A - Basic",
    prompt="""You are a Netflix content expert. Answer questions about the Netflix catalog accurately and concisely."""
)

result = answer_question(
    question="What are the top 3 highest-rated shows on Netflix?",
    model="gpt-4o",
    system_prompt=test_prompt
)

print(result)

## Creating the Evaluation Dataset

We define a set of questions spanning factual recall, comparison, recommendation, and analysis tasks. Each question has an expected answer grounded in the catalog data, giving the LLM-as-judge metrics a reference point.

In [ ]:
dataset_items = [
    {
        "question": "Which Netflix titles have the highest IMDb scores? List the top 3 with their scores.",
        "expected_answer": "The top 3 Netflix titles by IMDb score are: Dark (8.8), then Stranger Things and Black Mirror tied at 8.7. Dark is a German sci-fi thriller/mystery, Stranger Things is a sci-fi/horror/drama, and Black Mirror is a sci-fi/thriller anthology series."
    },
    {
        "question": "I loved Stranger Things and I'm looking for another sci-fi series with a mystery element. What would you recommend from the catalog and why?",
        "expected_answer": "I'd recommend Dark — it's a German sci-fi/thriller/mystery series with 3 seasons and the highest IMDb score in the catalog at 8.8. Like Stranger Things, it involves the disappearance of children in a small town with supernatural elements. Black Mirror (IMDb 8.7) is another great sci-fi option, though it's an anthology series with standalone episodes rather than a continuous mystery."
    },
    {
        "question": "What are all the films (not series) available in the catalog? Include their genres and runtimes.",
        "expected_answer": "There are 2 films in the catalog: 1) Glass Onion: A Knives Out Mystery (2022) — Mystery/Comedy/Crime, 139 minutes, rated PG-13, IMDb 7.1. 2) All Quiet on the Western Front (2022) — War/Drama/Action, 148 minutes, rated R, IMDb 7.8."
    },
    {
        "question": "Which show has the most total episodes and approximately how long would it take to watch them all?",
        "expected_answer": "The Crown has the most episodes at 60 total, with an average episode length of 58 minutes. Watching all episodes would take approximately 3,480 minutes, or about 58 hours."
    },
    {
        "question": "Compare Squid Game and Wednesday side by side. How are they similar and how do they differ?",
        "expected_answer": "Squid Game and Wednesday are both popular Netflix series from the early 2020s, each with 2 seasons and 16 episodes. Key differences: Squid Game is a TV-MA thriller/drama/action with an IMDb score of 8.0 about cash-strapped players in deadly games with a Korean cast (Lee Jung-jae, Park Hae-soo). Wednesday is a TV-14 comedy/horror/mystery scoring 8.1 on IMDb, following Wednesday Addams at Nevermore Academy, starring Jenna Ortega and Catherine Zeta-Jones."
    },
    {
        "question": "I want to watch something suitable for a teenager (TV-14 or PG-13 rating). What are my options?",
        "expected_answer": "There are 3 titles suitable for teenagers: 1) Stranger Things (TV-14) — a sci-fi/horror/drama series with 5 seasons and IMDb 8.7. 2) Wednesday (TV-14) — a comedy/horror/mystery series with 2 seasons and IMDb 8.1. 3) Glass Onion: A Knives Out Mystery (PG-13) — a mystery/comedy/crime film, 139 minutes, IMDb 7.1. All other titles are rated TV-MA or R."
    },
    {
        "question": "Which completed series would you recommend for a weekend binge-watch? Consider total watch time and quality rating.",
        "expected_answer": "The Queen's Gambit is ideal for a weekend binge — it's a completed limited series with just 7 episodes averaging 56 minutes each (about 6.5 hours total) and an excellent IMDb score of 8.5. Dark is another strong choice at 26 episodes averaging 52 minutes (about 22.5 hours) with the catalog's highest IMDb score of 8.8, though it requires a longer time commitment. The Crown (60 episodes, ~58 hours) and Stranger Things (42 episodes, ~35 hours) are also completed but need significantly more time."
    },
    {
        "question": "Has any title in the catalog won major awards? If so, which title and what did it win?",
        "expected_answer": "All Quiet on the Western Front (2022) won 4 Academy Awards including Best International Feature Film. It's a war/drama/action film starring Felix Kammerer about a young German soldier experiencing the horrors of World War I on the Western Front."
    },
    {
        "question": "What genres are most common across the Netflix catalog? Give a breakdown.",
        "expected_answer": "Drama is the most common genre, appearing in 8 out of 10 titles. Sci-Fi appears in 3 titles (Stranger Things, Black Mirror, Dark). Thriller appears in 3 titles (Squid Game, Black Mirror, Dark). Mystery appears in 3 titles (Wednesday, Glass Onion, Dark). Horror appears in 2 titles (Stranger Things, Wednesday). Comedy appears in 2 titles (Wednesday, Glass Onion). Historical appears in 2 titles (The Crown, Bridgerton). Action appears in 2 titles (Squid Game, All Quiet on the Western Front). War, Crime, Romance, and Biography each appear in 1 title."
    },
    {
        "question": "Summarize what Daniel Craig and Anya Taylor-Joy each star in on Netflix, with a brief description of each title.",
        "expected_answer": "Daniel Craig stars in Glass Onion: A Knives Out Mystery (2022), a PG-13 mystery/comedy/crime film (139 min, IMDb 7.1) where tech billionaire Miles Bron invites friends to his private Greek island and Detective Blanc investigates when someone turns up dead. Anya Taylor-Joy stars in The Queen's Gambit (2020), a TV-MA drama limited series (7 episodes, IMDb 8.5) about an orphaned chess prodigy in 1960s America who masters the game while struggling with addiction."
    }
]

In [ ]:
client = opik.Opik()

dataset = client.get_or_create_dataset(
    name="Netflix_Content_QA",
    description="Questions about Netflix shows and movies for evaluating content Q&A quality"
)

dataset.insert(dataset_items)

## Evaluating the Application

We evaluate 3 model + prompt configurations using industry-standard **LLM-as-judge** metrics from Opik, plus a heuristic similarity metric:

| Metric | What it measures | Scale |
|---|---|---|
| **Hallucination** | Does the response fabricate information not in the catalog? | 0 = factual, 1 = hallucination |
| **Answer Relevance** | How well does the response address the specific question? | 0–1 (higher = more relevant) |
| **Usefulness** | Overall helpfulness and quality of the response | 0–1 (higher = more useful) |
| **Levenshtein Ratio** | String similarity between the response and expected answer | 0–1 (higher = closer match) |


By default, all metrics will use gpt-5-nano as the LLM Judge. That said, you can also use another model or your own custom provider through the instructions [here](https://www.comet.com/docs/opik/evaluation/metrics/custom_model).

In [ ]:
#Optional: setup custom OppenAi compatible model provider to use as the LLM juge
# For OpenAI Compatible Providers:Automatically will use the OpenAI API key and base URL set at top of script
# model_name="openai/your-model-name"


In [ ]:
hallucination_metric = Hallucination()
answer_relevance_metric = AnswerRelevance()
usefulness_metric = Usefulness()
levenshtein_metric = LevenshteinRatio()

scoring_metrics = [hallucination_metric, answer_relevance_metric, usefulness_metric, levenshtein_metric]

dataset = client.get_dataset(name="Netflix_Content_QA")

In [ ]:
def make_evaluation_task(model, prompt):
    @opik.track(project_name=PROJECT_NAME)
    def evaluation_task(x):
        result = answer_question(
            question=x["question"],
            model=model,
            system_prompt=prompt
        )
        return {
            "output": result,
            "context": [NETFLIX_CATALOG]
        }
    return evaluation_task

### Experiment 1: gpt-4o-mini + Basic Prompt

The cheapest model paired with a prompt that actively encourages a divergent output style — markdown headers, bullet points, emojis, chatty commentary, and greetings. This should produce outputs that are **least similar** to the concise, factual expected answers.

In [ ]:
prompt_basic = opik.Prompt(
    name="Netflix Q&A",
    prompt="""You are a fun, enthusiastic Netflix fan chatbot! 🎬🍿 Answer questions with lots of energy and personality. Use markdown headers (##), bullet points, bold text, and emojis throughout your responses to make them visually engaging. Add personal opinions and editorial commentary about the shows. Give detailed, expansive answers — the more context and flavor the better! Always greet the user warmly and sign off with a fun closing line."""
)

res = evaluate(
    experiment_name="gpt-4o-mini | Basic Prompt",
    dataset=dataset,
    task=make_evaluation_task("gpt-4o-mini", prompt_basic),
    experiment_config={"model": "gpt-4o-mini", "prompt_version": "basic",  "training_experiment":"https://www.comet.com/examples/comet-example-nlp-text-classification/0b1c468a303b4e92910b12af4dee5fc7"},
    project_name=f"{PROJECT_NAME}-Experiments",
    prompt=prompt_basic,
    scoring_metrics=scoring_metrics,
    scoring_key_mapping={
        "input": "question",
        "output": "output",
        "context": "context",
        "reference": "expected_answer"
    }
)

### Experiment 2: gpt-4o + Standard Prompt

A more capable model with a structured prompt that defines a clear role, sets accuracy expectations, and provides formatting guidance.

In [ ]:
prompt_standard = opik.Prompt(
    name="Netflix Q&A",
    prompt="""You are a Netflix content assistant. Answer questions accurately using the provided catalog data.

Guidelines:
- Be concise — aim for 2-4 sentences when possible
- Cite specific details: titles, years, IMDb scores, ratings, episode counts
- Use numbered lists for multiple items
- Do not add greetings, sign-offs, or filler text
- Do not use markdown formatting or emojis"""
)

res = evaluate(
    experiment_name="gpt-4o | Standard Prompt",
    dataset=dataset,
    task=make_evaluation_task("gpt-4o", prompt_standard),
    experiment_config={"model": "gpt-4o", "prompt_version": "standard", "training_experiment":"https://www.comet.com/examples/comet-example-nlp-text-classification/95b3486025214ad090d94fd69bba7c10"},
    project_name=f"{PROJECT_NAME}-Experiments",
    prompt=prompt_standard,
    scoring_metrics=scoring_metrics,
    scoring_key_mapping={
        "input": "question",
        "output": "output",
        "context": "context",
        "reference": "expected_answer"
    }
)

### Experiment 3: gpt-4.1 + Expert Prompt

The most capable model with a detailed prompt that prescribes the exact output format — plain prose, inline numbered lists with "1)" format, em dashes, slash-separated genres, and a one-shot example. This should produce outputs that are **most similar** to the expected answers.

In [ ]:
prompt_expert = opik.Prompt(
    name="Netflix Q&A",
    prompt="""You are a Netflix content information specialist. Answer questions using ONLY the provided catalog data.

OUTPUT FORMAT RULES (follow exactly):
- Write in concise, information-dense prose. Target 2-4 sentences for most answers.
- For lists of multiple items, use this inline numbered format: "1) Title (Year) — Genre, details. 2) Title (Year) — Genre, details."
- Separate genres with slashes: "Sci-Fi/Horror/Drama"
- Always include specific data inline: titles, release years in parentheses, "IMDb X.X" for scores, episode counts, ratings, runtimes.
- Use em dashes (—) to separate parenthetical details within list items.
- Start with a direct factual answer. No greetings, sign-offs, opinions, or filler.
- Do NOT use any markdown formatting — no headers, no bold, no bullet points, no asterisks.
- When recommending, briefly explain WHY by connecting catalog attributes to the user's preferences.
- When comparing, state similarities first, then lead differences with "Key differences:" as a prefix.

EXAMPLE — Question: "What dramas are available with high ratings?"
Answer: "There are several highly-rated dramas: 1) Dark (2017) — Sci-Fi/Thriller/Mystery, 3 seasons, IMDb 8.8. 2) Stranger Things (2016) — Sci-Fi/Horror/Drama, 5 seasons, IMDb 8.7. 3) The Crown (2016) — Drama/Historical/Biography, 6 seasons, IMDb 8.6. All three are completed series rated TV-14 or TV-MA.\""""
)

res = evaluate(
    experiment_name="gpt-4.1 | Expert Prompt",
    dataset=dataset,
    task=make_evaluation_task("gpt-4.1", prompt_expert),
    experiment_config={"model": "gpt-4.1", "prompt_version": "expert", "training_experiment":"https://www.comet.com/examples/comet-example-nlp-text-classification/36570b6c9ad44475964227fcb8b18b0f"},
    project_name=f"{PROJECT_NAME}-Experiments",
    prompt=prompt_expert,
    scoring_metrics=scoring_metrics,
    scoring_key_mapping={
        "input": "question",
        "output": "output",
        "context": "context",
        "reference": "expected_answer"
    }
)